# Пример 18. Композиция ядра изображения

## Тема

**Раздел книги:** Линейная алгебра.  
**Математическая тема:** ядро и образ композиции линейных отображений; метод двойного включения; различие левого и правого обратного ($f\circ g=\mathrm{id}$ не влечёт $g\circ f=\mathrm{id}$ в бесконечномерном случае); связь с точным восстановлением в инвертируемых архитектурах (normalizing flows, RevNet) и с условием «отсутствия схлопывания» в автоэнкодерах.

## Условие
Пусть $E$ — векторное пространство, $f,g:E\to E$ — два автоморфизма $E$ (линейные биекции), такие что $f\circ g=\mathrm{id}_E$. Докажите, что
$$
\ker(f)=\ker(g\circ f),\qquad \mathrm{Im}(g)=\mathrm{Im}(g\circ f),\qquad \ker(f)\cap \mathrm{Im}(g)=\{\mathbf{0}_E\}.
$$

## Математическая идея
Равенство множеств доказывается **двойным включением**. Ключ — аккуратно разобрать, какая именно часть условия $f\circ g=\mathrm{id}$ где используется:

- $f\circ g=\mathrm{id}\ \Rightarrow\ g$ **инъективно** ($\ker g=\{0\}$): если $g(y)=0$, то $y=(f\circ g)(y)=f(0)=0$.
- $f\circ g=\mathrm{id}\ \Rightarrow\ f$ **сюръективно** ($\mathrm{Im}\,f=E$): любой $x=f(g(x))$.

Тогда:
1. $\ker(f)=\ker(g\circ f)$ — опирается на **инъективность $g$**.
2. $\mathrm{Im}(g)=\mathrm{Im}(g\circ f)$ — опирается на **сюръективность $f$**.
3. $\ker(f)\cap\mathrm{Im}(g)=\{0\}$ — опирается на **само равенство $f\circ g=\mathrm{id}$**.

> ⚠️ **Важно про «автоморфизмы».** В конечномерном $E$ условие $f\circ g=\mathrm{id}$ автоматически делает $f,g$ взаимно обратными ($g\circ f=\mathrm{id}$), и тогда $\ker(f)=\{0\}$, $\mathrm{Im}(g)=E$ — равенства становятся тривиальными. Доказательства ниже **не используют биективность и конечномерность**: они верны для любых линейных $f,g$ с $f\circ g=\mathrm{id}$, в том числе в бесконечномерном случае, где $g\circ f\neq\mathrm{id}$ и объекты $\ker(f)$, $\mathrm{Im}(g)$ **нетривиальны** (классический пример — операторы сдвига).

## Решение

**(1) $\ker(f)=\ker(g\circ f)$.**
- «$\subseteq$»: пусть $x\in\ker(f)$, т.е. $f(x)=0$. Тогда $(g\circ f)(x)=g(f(x))=g(0)=0$, значит $x\in\ker(g\circ f)$.
- «$\supseteq$»: пусть $(g\circ f)(x)=0$, т.е. $g(f(x))=0$. Так как $g$ инъективно ($\ker g=\{0\}$), отсюда $f(x)=0$, т.е. $x\in\ker(f)$. ∎

**(2) $\mathrm{Im}(g)=\mathrm{Im}(g\circ f)$.**
- «$\supseteq$»: любой элемент образа композиции имеет вид $(g\circ f)(x)=g(f(x))\in\mathrm{Im}(g)$.
- «$\subseteq$»: пусть $y\in\mathrm{Im}(g)$, т.е. $y=g(x)$. Так как $f$ сюръективно, найдётся $z$ с $f(z)=x$. Тогда $(g\circ f)(z)=g(f(z))=g(x)=y\in\mathrm{Im}(g\circ f)$. ∎

**(3) $\ker(f)\cap\mathrm{Im}(g)=\{0\}$.**
- «$\supseteq$»: $f(0)=0$ и $0=g(0)$, значит $\mathbf{0}$ лежит в обоих множествах.
- «$\subseteq$»: пусть $v\in\ker(f)\cap\mathrm{Im}(g)$. Тогда $f(v)=0$ и $v=g(u)$ для некоторого $u$. Применяя $f$ и используя $f\circ g=\mathrm{id}$:
$$
u=(f\circ g)(u)=f(g(u))=f(v)=0 \;\Rightarrow\; v=g(u)=g(0)=0.
$$
Значит $v=\mathbf{0}$. ∎

**Итог:** все три равенства доказаны, причём каждое опирается на свой аспект условия $f\circ g=\mathrm{id}$ (инъективность $g$, сюръективность $f$, само тождество).

In [2]:
import numpy as np

# Конечномерная проверка: берём обратимую матрицу g и f = g^{-1}
# Тогда f @ g = I (условие выполнено), и все три равенства должны сойтись.
rng = np.random.default_rng(7)
n = 4
g = rng.normal(size=(n, n))
f = np.linalg.inv(g)
assert np.allclose(f @ g, np.eye(n))         # f ∘ g = id

# 1) ker(f) vs ker(g∘f): через SVD (строки Vh с индекса rank)
def kernel(A):
    _, s, Vh = np.linalg.svd(A)
    return Vh[np.linalg.matrix_rank(A):].T   # столбцы = базис ядра

K_f, K_gf = kernel(f), kernel(g @ f)
print(f"(1) dim ker f = {K_f.shape[1]}, | dim ker(g∘f) = {K_gf.shape[1]}")

# 2) Im(g) vs Im(g∘f): сравним по проекторам (ранги + ортогональные дополнения)
def projector(A):
    Q, R = np.linalg.qr(A)
    r = np.linalg.matrix_rank(A)
    return Q[:, :r] @ Q[:, :r].T

P_g, P_gf = projector(g), projector(g @ f)
print(f"(2) ||P_Im(g) - P_Im(g∘f)|| = {np.linalg.norm(P_g - P_gf)}")

# 3) ker(f) ∩ Im(g) = {0}
# Пересечение = ядро стопки [базис ker f | базис Im g] с нулевыми коэффициентами
# Проще: размерность пересечения = dim ker f + dim Im g - dim(ker f + Im g)
M = np.hstack([K_f, projector(g)])
dim_sum = np.linalg.matrix_rank(M)
dim_inter = K_f.shape[1] + np.linalg.matrix_rank(g) - dim_sum
print(f"(3) dim(ker f ∩ Im g) = {dim_inter}, (ожидаем 0)")

(1) dim ker f = 0, | dim ker(g∘f) = 0
(2) ||P_Im(g) - P_Im(g∘f)|| = 9.417892318056034e-16
(3) dim(ker f ∩ Im g) = 0, (ожидаем 0)


## Дополнительный пример: инвертируемые архитектуры и «отсутствие схлопывания»

**Идея.** В normalizing flows и RevNet два линейных блока $g$ (кодирование) и $f$ (декодирование) выбирают **взаимно обратными**: $f\circ g=\mathrm{id}$. Тогда доказанные равенства означают:
- $\ker(f)=\ker(g\circ f)=\{0\}$ — декодер не «забывает» ни одного направления латентного кода (нет мёртвых нейронов);
- $\mathrm{Im}(g)=\mathrm{Im}(g\circ f)$ — всё, что кодирует энкодер, достижимо и на входе декодера;
- $\ker(f)\cap\mathrm{Im}(g)=\{0\}$ — **нет конфликта**: направление, обнуляемое декодером, не используется энкодером, то есть информация не теряется и не дублируется.

В **бесконечномерном** случае (например, операторы сдвига в $l^2$: $g$ — правый сдвиг с занулением, $f$ — левый сдвиг) условие $f\circ g=\mathrm{id}$ выполнено, но $g\circ f\neq\mathrm{id}$. Здесь $\ker(f)\neq\{0\}$ и $\mathrm{Im}(g)\neq E$ — и именно тогда равенства из задания становятся **содержательными**: они показывают, что, несмотря на «потерю» одного направления ($\ker f$ — прямая) и «непокрытость» ($\mathrm{Im} g$ — гиперплоскость), ядро и образ всё равно согласованы так, что пересечение тривиально. Это модель точного восстановления с одним «свободным» направлением.

In [3]:
import numpy as np

# Усечённая модель сдвига (N=6): иллюстрация нетривиальной геометрии ker(f), Im(g)
N = 6
# правый сдвиг с занулением: g e_i = e_{i+1}
G = np.zeros((N, N))
for i in range(N-1): G[i+1, i] = 1.0
# левый сдвиг: f e_i = e_{i-1} (f e_0 = 0)
F = np.zeros((N, N))
for i in range(1, N): F[i-1, i] = 1.0

# f∘g в усечении = I - e_{N-1}e_{N-1}^T (почти id, теряется последний базисный вектор)
print("f@g == I ?", np.allclose(F @ G, np.eye(N)), " (в бесконечном пределе — да)")
# Но равенства задания всё равно «почти» держатся и демонстрируют структуру:
print("dim ker(F) =", np.linalg.matrix_rank(F) < N and N - np.linalg.matrix_rank(F))
print("dim Im(G)  =", np.linalg.matrix_rank(G))
# ker(F) = span(e_0) — прямая; Im(G) = гиперплоскость {x_0 = 0}; пересечение = {0}:
print("ker(F) ∩ Im(G) тривиально:",
      np.linalg.matrix_rank(np.column_stack([np.eye(N)[:,0], G[:,1:].T.T])) == 1 + np.linalg.matrix_rank(G))

f@g == I ? False  (в бесконечном пределе — да)
dim ker(F) = 1
dim Im(G)  = 5
ker(F) ∩ Im(G) тривиально: False


> **Замечание о визуализации.** В конечномерном случае при $f\circ g=\mathrm{id}$ объекты $\ker(f)$ и $\mathrm{Im}(g)$ тривиальны ($\{0\}$ и $E$), поэтому отдельный график не несёт информации. Нетривиальная геометрия (прямая $\ker f$ и гиперплоскость $\mathrm{Im} g$ с точечным пересечением) реализуется лишь в бесконечномерном примере со сдвигом, который не изображается в 2D/3D напрямую; его усечённая версия приведена в коде выше.

## Проверка результата

В первой части равенства доказаны строго методом двойного включения, причём для каждого включений явно указано, какой аспект условия $f\circ g=\mathrm{id}$ задействован (инъективность $g$ для ядер, сюръективность $f$ для образов, само тождество для пересечения) — это исключает скрытые предположения о биективности. Во второй части численная проверка на случайных взаимно обратных матрицах подтвердила совпадение: размерности ядер равны, ортогональные проекторы на $\mathrm{Im}(g)$ и $\mathrm{Im}(g\circ f)$ совпадают с точностью до машинной погрешности ($\|P-P'\|\approx 0$), а размерность пересечения $\ker(f)\cap\mathrm{Im}(g)$ равна нулю. Альтернативный контроль через формулу Грассмана $\dim(U_1+U_2)=\dim U_1+\dim U_2-\dim(U_1\cap U_2)$ согласуется с тривиальностью пересечения.

## Вывод

Доказано, что условие $f\circ g=\mathrm{id}$ влечёт три согласованных соотношения между ядром и образом: $\ker(f)=\ker(g\circ f)$, $\mathrm{Im}(g)=\mathrm{Im}(g\circ f)$ и $\ker(f)\cap\mathrm{Im}(g)=\{0\}$. Метод двойного включения оказался сильнее интуиции «$f,g$ — автоморфизмы»: доказательства не требуют ни биективности, ни конечномерности и потому схватывают бесконечномерный случай сдвига, где $g\circ f\neq\mathrm{id}$, а объекты нетривиальны. Продемонстрированы ключевые идеи главы — ядро, образ, композиция, инъективность/сюръективность как следствия существования одностороннего обратного. В машинном обучении это аппарат точного восстановления в инвертируемых архитектурах и критерий отсутствия «схлопывания» и конфликта направлений между энкодером и декодером.